# Notebook 1: Data Loading

### StudentLife Dataset — Spring 2013
Load raw sensor, EMA, and survey data for the relevant participants.
All location data is dropped immediately on load for privacy.

In [14]:
import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

# Base path to dataset
DATA_DIR = Path("../data/studentlife")

# Study period
STUDY_START = pd.Timestamp("2013-03-25", tz="US/Eastern")
STUDY_END   = pd.Timestamp("2013-06-03", tz="US/Eastern")

# All participant IDs
UIDS = [f"u{i:02d}" for i in range(48)]

print("Imports complete.")
print(f"Study period: {STUDY_START.date()} to {STUDY_END.date()}")
print(f"Participants: {len(UIDS)}")

Imports complete.
Study period: 2013-03-25 to 2013-06-03
Participants: 48


### Privacy
All participant UIDs are hashed before any data is stored.
Location coordinates are dropped immediately on load and never stored.

In [15]:
def hash_uid(uid, salt="ddcm_2024"):
    """
    Irreversibly anonymise participant IDs using SHA-256 + salt.
    The raw UID is never stored downstream.
    """
    return hashlib.sha256(f"{salt}:{uid}".encode()).hexdigest()[:16]

# Build a lookup table: raw uid -> hashed uid
uid_map = {uid: hash_uid(uid) for uid in UIDS}

print("UID hashing ready.")
print(f"Example: u00 -> {uid_map['u00']}")

UID hashing ready.
Example: u00 -> cd6251b4de52453b


### Sensing Data: Activity
Raw file: one CSV per participant with Unix timestamp and activity inference code.
- 0 = stationary, 1 = walking, 2 = running, 3 = unknown
We aggregate to daily level: fraction of readings that are stationary.

In [16]:
activity_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "activity" / f"activity_{uid}.csv"
    if not fpath.exists():
        continue

    # Load the raw CSV and name the columns
    df = pd.read_csv(fpath, header=0, names=["timestamp", "activity"])
    
    # Convert timestamp to numeric, drop any rows that failed to parse
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])
    
    # Convert Unix timestamp to datetime in Eastern time, then extract just the date
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("US/Eastern")
    df["date"] = df["datetime"].dt.date

    # Aggregate to daily level: fraction of readings that are stationary (activity == 0)
    daily = df.groupby("date").apply(
        lambda g: pd.Series({
            "frac_stationary": (g["activity"] == 0).mean(),
            "n_activity_readings": len(g)
        })
    ).reset_index()

    daily["hashed_uid"] = uid_map[uid]
    activity_records.append(daily)

# Combine all participants into one dataframe
df_activity = pd.concat(activity_records, ignore_index=True)

# Filter to study period using tz-naive timestamps
df_activity["date"] = pd.to_datetime(df_activity["date"])
df_activity = df_activity[
    (df_activity["date"] >= pd.Timestamp("2013-03-25")) &
    (df_activity["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Activity loaded: {len(df_activity)} participant-days")
print(df_activity.head())

Activity loaded: 2262 participant-days
        date  frac_stationary  n_activity_readings        hashed_uid
0 2013-03-27         0.833979               7999.0  cd6251b4de52453b
1 2013-03-28         0.895264               7581.0  cd6251b4de52453b
2 2013-03-29         0.839825               7985.0  cd6251b4de52453b
3 2013-03-30         0.878117               4652.0  cd6251b4de52453b
4 2013-03-31         0.837083               8090.0  cd6251b4de52453b


### Sensing Data: Phone Lock
Raw file: one CSV per participant with start and end Unix timestamps of lock periods.
We sum total lock duration per day as a proxy for inactivity / sleep.

In [17]:
phonelock_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "phonelock" / f"phonelock_{uid}.csv"
    if not fpath.exists():
        continue

    # Load the raw CSV and name the columns
    df = pd.read_csv(fpath, header=0, names=["start_ts", "end_ts"])
    
    # Convert to numeric, drop any rows that failed to parse
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    
    # Calculate how long the phone was locked in hours
    df["duration_hours"] = (df["end_ts"] - df["start_ts"]) / 3600
    
    # Convert start timestamp to a date (Eastern time)
    df["date"] = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("US/Eastern").dt.date

    # Sum all lock periods within each day
    daily = df.groupby("date")["duration_hours"].sum().reset_index()
    daily.columns = ["date", "phonelock_hours"]
    daily["hashed_uid"] = uid_map[uid]
    phonelock_records.append(daily)

df_phonelock = pd.concat(phonelock_records, ignore_index=True)

# Filter to study period using tz-naive timestamps
df_phonelock["date"] = pd.to_datetime(df_phonelock["date"])
df_phonelock = df_phonelock[
    (df_phonelock["date"] >= pd.Timestamp("2013-03-25")) &
    (df_phonelock["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Phone lock loaded: {len(df_phonelock)} participant-days")
print(df_phonelock.head())

Phone lock loaded: 1957 participant-days
        date  phonelock_hours        hashed_uid
0 2013-03-27         9.845000  cd6251b4de52453b
1 2013-03-28         2.723056  cd6251b4de52453b
2 2013-03-29        11.428056  cd6251b4de52453b
3 2013-03-30         9.818333  cd6251b4de52453b
4 2013-03-31        16.463333  cd6251b4de52453b


### Sensing Data: Conversation
Raw file: one CSV per participant with start and end Unix timestamps of detected conversation periods.
We sum total conversation duration per day in hours as a proxy for social activity.

In [18]:
conversation_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "conversation" / f"conversation_{uid}.csv"
    if not fpath.exists():
        continue

    # Load the raw CSV and name the columns
    df = pd.read_csv(fpath, header=0, names=["start_ts", "end_ts"])
    
    # Convert to numeric, drop any rows that failed to parse
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    
    # Calculate how long each conversation period lasted in hours
    df["duration_hours"] = (df["end_ts"] - df["start_ts"]) / 3600
    
    # Convert start timestamp to a date (Eastern time)
    df["date"] = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("US/Eastern").dt.date

    # Sum all conversation periods within each day
    daily = df.groupby("date")["duration_hours"].sum().reset_index()
    daily.columns = ["date", "conversation_hours"]
    daily["hashed_uid"] = uid_map[uid]
    conversation_records.append(daily)

df_conversation = pd.concat(conversation_records, ignore_index=True)

# Filter to study period using tz-naive timestamps
df_conversation["date"] = pd.to_datetime(df_conversation["date"])
df_conversation = df_conversation[
    (df_conversation["date"] >= pd.Timestamp("2013-03-25")) &
    (df_conversation["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Conversation loaded: {len(df_conversation)} participant-days")
print(df_conversation.head())

Conversation loaded: 2127 participant-days
        date  conversation_hours        hashed_uid
0 2013-03-27            7.073056  cd6251b4de52453b
1 2013-03-28            6.414444  cd6251b4de52453b
2 2013-03-29            8.581389  cd6251b4de52453b
3 2013-03-30           10.140000  cd6251b4de52453b
4 2013-03-31            6.927500  cd6251b4de52453b


### EMA Data: Mood
Raw file: one JSON per participant. Each entry has:
- happy (1–4), sad (1–4), happyornot (1–2), sadornot (1–2), resp_time, location
Location is dropped immediately. 
Mood score = happy - sad (range -3 to +3), shifted to 1–7 for interpretability.
We keep one response per participant per day (the last response of the day).

In [19]:
mood_records = []

for uid in UIDS:
    fpath = DATA_DIR / "EMA" / "response" / "Mood" / f"Mood_{uid}.json"
    if not fpath.exists():
        continue

    # Load the JSON file for this participant
    with open(fpath) as f:
        entries = json.load(f)

    for entry in entries:
        try:
            happy = int(entry.get("happy", None))
            sad   = int(entry.get("sad", None))
            resp_time = entry.get("resp_time")
            if resp_time is None:
                continue
            
            # Mood score: happy - sad shifted to positive scale (1-7)
            # happy and sad are each 1-4, so raw range is -3 to +3, +4 shifts to 1-7
            mood_score = (happy - sad) + 4
            
            mood_records.append({
                "hashed_uid": uid_map[uid],
                "resp_time": resp_time,
                "mood_score": mood_score,
                "happy_raw": happy,
                "sad_raw": sad
                # location intentionally not stored — dropped for privacy
            })
        except (TypeError, ValueError):
            # Skip malformed entries missing happy/sad fields
            continue

# Combine all participants into one dataframe
df_mood = pd.DataFrame(mood_records)

# Convert Unix timestamp to datetime in Eastern time
df_mood["datetime"] = pd.to_datetime(df_mood["resp_time"], unit="s", utc=True).dt.tz_convert("US/Eastern")
df_mood["date"] = df_mood["datetime"].dt.normalize()

# Filter to study period using tz-aware timestamps to match normalized datetime column
df_mood = df_mood[
    (df_mood["date"] >= STUDY_START.normalize()) &
    (df_mood["date"] <= STUDY_END.normalize())
]

# Keep only the last response per person per day (some participants responded multiple times)
df_mood = df_mood.sort_values("datetime").groupby(["hashed_uid", "date"]).last().reset_index()

# Drop intermediate columns, keep only what we need
df_mood = df_mood[["hashed_uid", "date", "mood_score", "happy_raw", "sad_raw"]]

print(f"Mood EMA loaded: {len(df_mood)} participant-days")
print(df_mood.head())

Mood EMA loaded: 124 participant-days
         hashed_uid                      date  mood_score  happy_raw  sad_raw
0  02b34100f41b4332 2013-04-29 00:00:00-04:00           3          1        2
1  02b34100f41b4332 2013-05-05 00:00:00-04:00           3          2        3
2  02b34100f41b4332 2013-05-15 00:00:00-04:00           1          1        4
3  02b34100f41b4332 2013-05-16 00:00:00-04:00           1          1        4
4  02b34100f41b4332 2013-05-17 00:00:00-04:00           2          1        3


### EMA Data: Stress
Raw file: one JSON per participant. Valid entries have a "level" field (1–5).
Early malformed entries with "null" as key are skipped.
Location is dropped. One response per participant per day (last of day).

In [20]:
stress_records = []

for uid in UIDS:
    fpath = DATA_DIR / "EMA" / "response" / "Stress" / f"Stress_{uid}.json"
    if not fpath.exists():
        continue

    # Load the JSON file for this participant
    with open(fpath) as f:
        entries = json.load(f)

    for entry in entries:
        try:
            # Skip malformed early entries that have "null" as a key instead of "level"
            level = entry.get("level")
            if level is None:
                continue
            resp_time = entry.get("resp_time")
            if resp_time is None:
                continue

            stress_records.append({
                "hashed_uid": uid_map[uid],
                "resp_time": resp_time,
                "stress_level": int(level)
                # location intentionally not stored — dropped for privacy
            })
        except (TypeError, ValueError):
            # Skip any other malformed entries
            continue

# Combine all participants into one dataframe
df_stress = pd.DataFrame(stress_records)

# Convert Unix timestamp to datetime in Eastern time
df_stress["datetime"] = pd.to_datetime(df_stress["resp_time"], unit="s", utc=True).dt.tz_convert("US/Eastern")
df_stress["date"] = df_stress["datetime"].dt.normalize()

# Filter to study period — date is tz-aware so STUDY_START/END match
df_stress = df_stress[
    (df_stress["date"] >= STUDY_START.normalize()) &
    (df_stress["date"] <= STUDY_END.normalize())
]

# Keep only the last response per person per day
df_stress = df_stress.sort_values("datetime").groupby(["hashed_uid", "date"]).last().reset_index()

# Drop intermediate columns, keep only what we need
df_stress = df_stress[["hashed_uid", "date", "stress_level"]]

print(f"Stress EMA loaded: {len(df_stress)} participant-days")
print(df_stress.head())

Stress EMA loaded: 932 participant-days
         hashed_uid                      date  stress_level
0  02b34100f41b4332 2013-03-28 00:00:00-04:00             1
1  02b34100f41b4332 2013-03-30 00:00:00-04:00             2
2  02b34100f41b4332 2013-04-11 00:00:00-04:00             3
3  02b34100f41b4332 2013-04-14 00:00:00-04:00             3
4  02b34100f41b4332 2013-04-15 00:00:00-04:00             3


### EMA Data: Sleep
Raw file: one JSON per participant. Valid entries have "hour" (hours slept),
"rate" (sleep quality 1–4), and "social" fields.
Early malformed entries with "null" as key are skipped.
Location is dropped. One response per participant per day (last of day).

In [23]:
sleep_records = []

for uid in UIDS:
    fpath = DATA_DIR / "EMA" / "response" / "Sleep" / f"Sleep_{uid}.json"
    if not fpath.exists():
        continue

    # Load the JSON file for this participant
    with open(fpath) as f:
        entries = json.load(f)

    for entry in entries:
        try:
            # Skip malformed early entries that have "null" as a key instead of "hour"
            hour = entry.get("hour")
            rate = entry.get("rate")
            if hour is None:
                continue
            resp_time = entry.get("resp_time")
            if resp_time is None:
                continue

            sleep_records.append({
                "hashed_uid": uid_map[uid],
                "resp_time": resp_time,
                "sleep_hours": int(hour),          # self-reported hours slept
                "sleep_quality": int(rate) if rate else None  # quality rating 1-4, if present
                # location intentionally not stored — dropped for privacy
            })
        except (TypeError, ValueError):
            # Skip any other malformed entries
            continue

# Combine all participants into one dataframe
df_sleep = pd.DataFrame(sleep_records)

# Convert Unix timestamp to datetime in Eastern time
df_sleep["datetime"] = pd.to_datetime(df_sleep["resp_time"], unit="s", utc=True).dt.tz_convert("US/Eastern")
df_sleep["date"] = df_sleep["datetime"].dt.normalize()

# Filter to study period — date is tz-aware so STUDY_START/END match
df_sleep = df_sleep[
    (df_sleep["date"] >= STUDY_START.normalize()) &
    (df_sleep["date"] <= STUDY_END.normalize())
]

# Keep only the last response per person per day
df_sleep = df_sleep.sort_values("datetime").groupby(["hashed_uid", "date"]).last().reset_index()

# Drop intermediate columns, keep only what we need
df_sleep = df_sleep[["hashed_uid", "date", "sleep_hours", "sleep_quality"]]

print(f"Sleep EMA loaded: {len(df_sleep)} participant-days")
print(df_sleep.head())

Sleep EMA loaded: 900 participant-days
         hashed_uid                      date  sleep_hours  sleep_quality
0  02b34100f41b4332 2013-03-27 00:00:00-04:00            6              2
1  02b34100f41b4332 2013-03-28 00:00:00-04:00            7              2
2  02b34100f41b4332 2013-03-29 00:00:00-04:00            7              3
3  02b34100f41b4332 2013-03-30 00:00:00-04:00            7              2
4  02b34100f41b4332 2013-03-31 00:00:00-04:00            6              3


### Survey Data: PHQ-9
Raw file: one CSV with all participants. Text responses converted to numeric scores.
PHQ-9 scoring: Not at all=0, Several days=1, More than half the days=2, Nearly every day=3.
Total score range: 0–27.

In [24]:
# Mapping from text responses to numeric scores (standard PHQ-9 scoring)
phq9_map = {
    "Not at all": 0,
    "Several days": 1,
    "More than half the days": 2,
    "Nearly every day": 3
}

# The 9 symptom question columns — each scored 0-3
phq9_cols = [
    "Little interest or pleasure in doing things",
    "Feeling down, depressed, hopeless.",
    "Trouble falling or staying asleep, or sleeping too much.",
    "Feeling tired or having little energy",
    "Poor appetite or overeating",
    "Feeling bad about yourself or that you are a failure or have let yourself or your family down",
    "Trouble concentrating on things, such as reading the newspaper or watching television",
    "Moving or speaking so slowly that other people could have noticed. Or the opposite being so figety or restless that you have been moving around a lot more than usual",
    "Thoughts that you would be better off dead, or of hurting yourself"
]

# Load the raw CSV — one row per participant per survey type (pre/post)
df_phq9_raw = pd.read_csv(DATA_DIR / "survey" / "PHQ-9.csv")

# Replace text responses with numeric scores using the mapping above
df_phq9_raw[phq9_cols] = df_phq9_raw[phq9_cols].replace(phq9_map)

# Sum all 9 items to get total PHQ-9 score (range 0-27)
df_phq9_raw["phq9_score"] = df_phq9_raw[phq9_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

# Hash the raw UID for privacy
df_phq9_raw["hashed_uid"] = df_phq9_raw["uid"].map(uid_map)

# Keep only the columns we need: hashed ID, pre/post type, and total score
df_phq9 = df_phq9_raw[["hashed_uid", "type", "phq9_score"]].copy()

print(f"PHQ-9 loaded: {len(df_phq9)} records")
print(df_phq9.head())

PHQ-9 loaded: 84 records
         hashed_uid type  phq9_score
0  cd6251b4de52453b  pre           2
1  26119daf181335b2  pre           5
2  37985280ff34ce04  pre          13
3  4fc7d2e42c31e91b  pre           2
4  5c373475c11275cb  pre           6


### Save Loaded Data
Save each dataframe as a CSV to the data/ folder for use in the next notebook.
These files contain no raw UIDs and no location data.

In [25]:
# Create the processed data directory if it doesn't exist
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save each cleaned dataframe as a CSV for use in the next notebook
df_activity.to_csv(OUTPUT_DIR / "activity.csv", index=False)
df_phonelock.to_csv(OUTPUT_DIR / "phonelock.csv", index=False)
df_conversation.to_csv(OUTPUT_DIR / "conversation.csv", index=False)
df_mood.to_csv(OUTPUT_DIR / "mood.csv", index=False)
df_stress.to_csv(OUTPUT_DIR / "stress.csv", index=False)
df_sleep.to_csv(OUTPUT_DIR / "sleep.csv", index=False)
df_phq9.to_csv(OUTPUT_DIR / "phq9.csv", index=False)

print("All files saved to data/processed/")
print("\nRow counts:")
for name, df in [("activity", df_activity), ("phonelock", df_phonelock),
                  ("conversation", df_conversation), ("mood", df_mood),
                  ("stress", df_stress), ("sleep", df_sleep), ("phq9", df_phq9)]:
    print(f"  {name}: {len(df)} rows")

All files saved to data/processed/

Row counts:
  activity: 2262 rows
  phonelock: 1957 rows
  conversation: 2127 rows
  mood: 124 rows
  stress: 932 rows
  sleep: 900 rows
  phq9: 84 rows
